In [8]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt

In [9]:
X_train = np.load("../data/processed/X_train.npy")
X_test  = np.load("../data/processed/X_test.npy")
y_train = np.load("../data/processed/y_train.npy")
y_test  = np.load("../data/processed/y_test.npy")

print(X_train.shape, y_train.shape)
print(f"AD in train: {y_train.sum()}, Control: {(y_train==0).sum()}")

(138, 5000) (138,)
AD in train: 26, Control: 112


In [10]:
X_train_t = torch.FloatTensor(X_train)
X_test_t  = torch.FloatTensor(X_test)
y_train_t = torch.LongTensor(y_train)
y_test_t  = torch.LongTensor(y_test)

class_counts = np.bincount(y_train)
weights      = 1.0 / class_counts
sample_weights = torch.FloatTensor([weights[y] for y in y_train])

sampler = WeightedRandomSampler(
    weights     = sample_weights,
    num_samples = len(sample_weights),
    replacement = True
)

train_ds     = TensorDataset(X_train_t, y_train_t)
test_ds      = TensorDataset(X_test_t,  y_test_t)

train_loader = DataLoader(train_ds, batch_size=16, sampler=sampler)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

Using: cuda


In [11]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.6),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        return self.net(x)

In [12]:
def train_model(model, train_loader, val_loader, epochs=150, lr=1e-3, patience=20):
    model = model.to(device)

    class_weights_tensor = torch.FloatTensor([1.0, 4.24]).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)

    best_auc        = 0
    best_weights    = None
    patience_counter = 0
    history         = {"loss": [], "auc": []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        model.eval()
        all_probs, all_labels = [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                out   = model(X_batch.to(device))
                probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
                all_probs.extend(probs)
                all_labels.extend(y_batch.numpy())

        auc = roc_auc_score(all_labels, all_probs)
        scheduler.step(1 - auc)

        history["loss"].append(avg_loss)
        history["auc"].append(auc)

        if auc > best_auc:
            best_auc     = auc
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | AUC: {auc:.4f} | Best: {best_auc:.4f} | Patience: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(best_weights)
    return model, history

In [15]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

def run_kfold(X, y, k=5, top_k=5000, epochs=150, lr=1e-3):
    skf       = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    fold_aucs = []
    fold_reports = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\nFold {fold+1}/{k}  |  AD in val: {y[val_idx].sum()}  |  Control in val: {(y[val_idx]==0).sum()}")

        X_tr_raw, X_val_raw = X[train_idx], X[val_idx]
        y_tr,     y_val     = y[train_idx], y[val_idx]

        variances = X_tr_raw.var(axis=0)
        top_idx   = np.argsort(variances)[::-1][:top_k]
        X_tr_sel  = X_tr_raw[:, top_idx]
        X_val_sel = X_val_raw[:, top_idx]

        scaler    = StandardScaler()
        X_tr_sel  = scaler.fit_transform(X_tr_sel)
        X_val_sel = scaler.transform(X_val_sel)

        X_tr_t  = torch.FloatTensor(X_tr_sel)
        X_val_t = torch.FloatTensor(X_val_sel)
        y_tr_t  = torch.LongTensor(y_tr)
        y_val_t = torch.LongTensor(y_val)

        class_counts   = np.bincount(y_tr)
        weights        = 1.0 / class_counts
        sample_weights = torch.FloatTensor([weights[yi] for yi in y_tr])
        sampler        = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

        train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=16, sampler=sampler)
        val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=16, shuffle=False)

        model, history = train_model(MLP(input_dim=top_k), train_loader, val_loader, epochs=epochs, lr=lr)

        model.eval()
        all_probs, all_preds, all_labels = [], [], []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                out   = model(X_batch.to(device))
                probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
                preds = out.argmax(dim=1).cpu().numpy()
                all_probs.extend(probs)
                all_preds.extend(preds)
                all_labels.extend(y_batch.numpy())

        fold_auc = roc_auc_score(all_labels, all_probs)
        fold_aucs.append(fold_auc)

        print(f"Fold {fold+1} AUC: {fold_auc:.4f}")
        print(classification_report(all_labels, all_preds, target_names=["Control", "Alzheimer's"]))
        print(f"Confusion Matrix:\n{confusion_matrix(all_labels, all_preds)}")

    print(f"\nMean AUC : {np.mean(fold_aucs):.4f}")
    print(f"Std  AUC : {np.std(fold_aucs):.4f}")
    return fold_aucs


In [16]:
X_full = np.load("../data/processed/X_raw.npy")
y_full = np.load("../data/processed/y_raw.npy")

fold_aucs = run_kfold(X_full, y_full, k=5, top_k=5000, epochs=150, lr=1e-3)


Fold 1/5  |  AD in val: 7  |  Control in val: 28
Epoch  20 | Loss: 0.0827 | AUC: 0.9235 | Best: 0.9592 | Patience: 3/20
Early stopping at epoch 37
Fold 1 AUC: 0.9592
              precision    recall  f1-score   support

     Control       0.96      0.86      0.91        28
 Alzheimer's       0.60      0.86      0.71         7

    accuracy                           0.86        35
   macro avg       0.78      0.86      0.81        35
weighted avg       0.89      0.86      0.87        35

Confusion Matrix:
[[24  4]
 [ 1  6]]

Fold 2/5  |  AD in val: 7  |  Control in val: 28
Epoch  20 | Loss: 0.0567 | AUC: 1.0000 | Best: 1.0000 | Patience: 11/20
Early stopping at epoch 29
Fold 2 AUC: 1.0000
              precision    recall  f1-score   support

     Control       1.00      0.64      0.78        28
 Alzheimer's       0.41      1.00      0.58         7

    accuracy                           0.71        35
   macro avg       0.71      0.82      0.68        35
weighted avg       0.88      